Day 1 Complete Setup - PROF Baseline GRPO
==========================================
Complete implementation of:
- Unsloth GRPO setup
- Qwen2.5-Math-PRM-7B loading (4-bit)
- PROF Algorithm 1 filtering
- Smoke test

Copy-paste this entire file into Kaggle notebook and run cell-by-cell.
GPU: P100 (16GB), Internet: ON, Persistence: ON

In [1]:
# ============================================================================
# SECTION 1: DEPENDENCIES \u0026 ENVIRONMENT SETUP
# ============================================================================

print("=" * 60)
print("SECTION 1: Installing dependencies...")
print("=" * 60)

SECTION 1: Installing dependencies...


In [1]:
import sys
print(f"Python: {sys.version}")

# Step 1: Install core dependencies first (these are fast)
print("\n=== Step 1/4: Installing core packages ===")
!pip install -U datasets scipy transformers accelerate peft bitsandbytes

# Step 2: Install TRL with specific version
print("\n=== Step 2/4: Installing TRL ===")
!pip install "trl<0.9.0"

# Step 3: Install xformers (this can be slow, but shows progress)
print("\n=== Step 3/4: Installing xformers ===")
!pip install "xformers<0.0.27"

# Step 4: Install Unsloth (from PyPI, faster than git)
print("\n=== Step 4/4: Installing Unsloth ===")
# Try PyPI first (faster)
try:
    !pip install unsloth
    print("✅ Unsloth installed from PyPI")
except:
    # Fallback to git if needed (with progress visible)
    print("⚠️ PyPI failed, installing from git (this may take 5-10 min)")
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("\n" + "="*60)
print("✅ ALL DEPENDENCIES INSTALLED")
print("="*60)

# Verify installations
import torch
import transformers
import unsloth
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
import torch.nn.functional as F
from datasets import load_dataset
import re
import time
from datetime import datetime

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\nVersions:")
print(f"  PyTorch: {torch.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  Unsloth: {unsloth.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✅ Ready to proceed to Section 2")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

=== Step 1/4: Installing core packages ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 62.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 112.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 102.6 MB/s eta 0:00:0000:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled sci

/tmp/ipykernel_57/1119854215.py:34: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

Versions:
  PyTorch: 2.10.0+cu128
  Transformers: 5.5.0
  Unsloth: 2026.5.2
  CUDA available: True
  GPU: Tesla T4
  VRAM: 15.64 GB

✅ Ready to proceed to Section 2


In [2]:
# ============================================================================
# SECTION 2: LOAD POLICY MODEL (Qwen2.5-Math-1.5B)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 2: Loading policy model...")
print("=" * 60)

from unsloth import FastLanguageModel

# Load Qwen2.5-Math-1.5B in bf16 (should use ~6-7GB)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Math-1.5B",
    max_seq_length=4096,  # Per Ye et al. Section 4.1
    dtype=torch.bfloat16,  # P100 safe (no FP16 issues)
    load_in_4bit=False,    # Full precision training
)

print(f"✅ Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Add LoRA adapters (reduces memory, faster training)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,              # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=3407,
)

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")


SECTION 2: Loading policy model...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[unsloth_zoo.log|WARNING]Device does not support bfloat16. Will change to float16.


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

✅ Model loaded. VRAM used: 3.10 GB


Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Trainable: 18,464,768 / 1,562,179,072 (1.18%)


In [45]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [46]:
# ============================================================================
# SECTION 3: LOAD PRM MODEL (Qwen2.5-Math-PRM-7B in bf16)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 3: Loading PRM model...")
print("=" * 60)

def load_prm_model(model_name="Qwen/Qwen2.5-Math-PRM-7B", device="auto"):
    """
    Load PRM in bf16 for inference (4-bit was causing NaN logits).
    Per Ye et al.: frozen PRM, no training, just scoring.

    Note: Using full bf16 instead of 4-bit quantization to avoid NaN outputs.
    This uses ~7GB VRAM instead of ~2-3GB, but is necessary for correct scoring.
    """
    print(f"  Loading PRM tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # CRITICAL FIX: The Qwen2RMConfig in config.json is missing pad_token_id
    # This causes AttributeError at line 814 of modeling_qwen2_rm.py
    # We MUST patch the config before model loading
    print(f"  Loading and patching config (pad_token_id missing from Qwen2RMConfig)...")
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

    # Set pad_token_id to eos_token_id (standard for Qwen models)
    # From config.json: eos_token_id = 151645
    config.pad_token_id = config.eos_token_id
    print(f"     Patched: config.pad_token_id = {config.pad_token_id}")

    # Load model with patched config
    model = AutoModel.from_pretrained(
        model_name,
        config=config,  # Pass patched config
        device_map=device,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    ).eval()

    # Freeze all parameters (no gradients needed)
    for param in model.parameters():
        param.requires_grad = False

    print(f"  PRM model loaded successfully")
    return model, tokenizer

# Load PRM
prm_model, prm_tokenizer = load_prm_model()
print(f"✅ PRM loaded. Total VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# DEBUG: Check model architecture and weights
print(f"\n[CRITICAL DEBUG] Model architecture check...")
print(f"  Model type: {type(prm_model).__name__}")
print(f"  Model has 'score' module: {hasattr(prm_model, 'score')}")
print(f"  Model has 'lm_head' module: {hasattr(prm_model, 'lm_head')}")
print(f"  Top-level modules: {[name for name, _ in prm_model.named_children()]}")

print(f"\n[CRITICAL DEBUG] Checking model weights for NaN...")
nan_params = []
total_params_checked = 0
for name, param in prm_model.named_parameters():
    total_params_checked += 1
    if torch.isnan(param).any():
        nan_params.append(name)
        print(f"  ❌ NaN found in: {name}")
    if total_params_checked <= 5:  # Show first 5 param stats
        print(f"  Param {name}: shape={param.shape}, mean={param.mean().item():.4f}, std={param.std().item():.4f}")

if nan_params:
    print(f"\n⚠️  CRITICAL: {len(nan_params)} parameters contain NaN!")
    print(f"   This explains the NaN logits - model weights are corrupted!")
else:
    print(f"  ✅ All {total_params_checked} model weights are valid (no NaN)")



SECTION 3: Loading PRM model...
  Loading PRM tokenizer...
  Loading and patching config (pad_token_id missing from Qwen2RMConfig)...
     Patched: config.pad_token_id = 151645


Loading weights:   0%|          | 0/342 [00:00<?, ?it/s]

Qwen2ForProcessRewardModel LOAD REPORT from: Qwen/Qwen2.5-Math-PRM-7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  PRM model loaded successfully
✅ PRM loaded. Total VRAM used: 10.36 GB

[CRITICAL DEBUG] Model architecture check...
  Model type: Qwen2ForProcessRewardModel
  Model has 'score' module: True
  Model has 'lm_head' module: False
  Top-level modules: ['model', 'score']

[CRITICAL DEBUG] Checking model weights for NaN...
  Param model.embed_tokens.weight: shape=torch.Size([152064, 3584]), mean=-0.0000, std=0.0165
  Param model.layers.0.self_attn.q_proj.weight: shape=torch.Size([3584, 3584]), mean=-0.0000, std=0.0254
  Param model.layers.0.self_attn.q_proj.bias: shape=torch.Size([3584]), mean=0.1069, std=3.1094
  Param model.layers.0.self_attn.k_proj.weight: shape=torch.Size([512, 3584]), mean=-0.0000, std=0.0330
  Param model.layers.0.self_attn.k_proj.bias: shape=torch.Size([512]), mean=1.4219, std=28.0000
  ✅ All 342 model weights are valid (no NaN)


In [47]:
# ============================================================================
# SECTION 4: IMPLEMENT STEP-WISE PRM SCORING
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 4: Implementing PRM scoring functions...")
print("=" * 60)

def make_step_rewards(logits, token_masks):
    """
    Extract step-wise reward scores from PRM logits.

    From Qwen2.5-Math-PRM-7B model card implementation.

    Args:
        logits: Model output logits of shape (batch_size, seq_length, 2)
        token_masks: Boolean mask marking step separator positions

    Returns:
        List of lists where each inner list contains step rewards [0-1]
    """
    probabilities = F.softmax(logits, dim=-1)
    probabilities = probabilities * token_masks.unsqueeze(-1)

    all_scores_res = []
    for i in range(probabilities.size(0)):
        sample = probabilities[i]
        # Extract positive class probability at step separators
        positive_probs = sample[sample != 0].view(-1, 2)[:, 1]
        non_zero_elements_list = positive_probs.cpu().tolist()
        all_scores_res.append(non_zero_elements_list)
    return all_scores_res


def compute_step_rewards(response_text, model, tokenizer):
    """
    Compute step-wise PRM scores for a response.

    Per Ye et al. Section 3.1: Steps delimited by newlines.

    Args:
        response_text: Full response text from policy model
        model: Loaded PRM model
        tokenizer: Corresponding tokenizer

    Returns:
        step_rewards: List of reward scores for each step [0-1]
    """
    # Split response into steps by double newline (Ye et al. approach)
    steps = response_text.split('\n\n')

    # Filter out empty steps
    steps = [s.strip() for s in steps if s.strip()]

    if not steps:
        return []

    # Format for chat template with <extra_0> step markers
    # This is how Qwen2.5-Math-PRM expects input
    messages = [
        {"role": "system", "content": "Please reason step by step."},
        {"role": "user", "content": "Solve this problem."},
        {"role": "assistant", "content": "<extra_0>".join(steps) + "<extra_0>"},
    ]

    # Apply chat template and tokenize
    conversation_str = prm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize with attention_mask (CRITICAL for avoiding NaN)
    tokenized = prm_tokenizer(
        conversation_str,
        return_tensors="pt",
        padding=False,  # No padding needed for single sequence
    )
    input_ids = tokenized['input_ids'].to(prm_model.device)
    attention_mask = tokenized['attention_mask'].to(prm_model.device)

    # Forward pass through PRM (no gradients)
    # CRITICAL: Must pass attention_mask to avoid NaN in attention computation
    with torch.no_grad():
        outputs = prm_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )

    # Debug: Check output structure
    print(f"  [DEBUG] Output type: {type(outputs)}")
    print(f"  [DEBUG] Output keys: {outputs.keys() if hasattr(outputs, 'keys') else 'N/A'}")

    # Try to get logits - different models have different output structures
    if hasattr(outputs, 'logits'):
        logits = outputs.logits
        print(f"  [DEBUG] Using outputs.logits")
    elif isinstance(outputs, tuple) and len(outputs) > 0:
        logits = outputs[0]
        print(f"  [DEBUG] Using outputs[0]")
    else:
        logits = outputs
        print(f"  [DEBUG] Using outputs directly")

    print(f"  [DEBUG] Logits shape: {logits.shape}")
    print(f"  [DEBUG] Logits dtype: {logits.dtype}")
    print(f"  [DEBUG] Logits device: {logits.device}")
    print(f"  [DEBUG] Logits contains NaN: {torch.isnan(logits).any()}")
    print(f"  [DEBUG] Logits contains Inf: {torch.isinf(logits).any()}")
    print(f"  [DEBUG] Logits min/max: {logits.min().item():.4f} / {logits.max().item():.4f}")

    # Extract rewards at step separator positions
    step_sep_id = prm_tokenizer.encode("<extra_0>")[0]
    token_masks = (input_ids == step_sep_id)

    print(f"  [DEBUG] Step separator ID: {step_sep_id}")
    print(f"  [DEBUG] Number of step separators found: {token_masks.sum().item()}")

    step_rewards = make_step_rewards(logits, token_masks)

    return step_rewards[0] if step_rewards else []

# Test PRM scoring - using format closer to official example
print("\n[TEST] Testing PRM scoring with official example format...")
test_data = {
    "system": "Please reason step by step.",
    "query": "What is 2+2?",
    "response": [
        "Let me solve this step by step.",
        "Step 1: Add the two numbers together.",
        "Step 2: 2 + 2 = 4.",
        "The answer is 4."
    ]
}

test_messages = [
    {"role": "system", "content": test_data['system']},
    {"role": "user", "content": test_data['query']},
    {"role": "assistant", "content": "<extra_0>".join(test_data['response']) + "<extra_0>"},
]

test_conversation = prm_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)

print(f"  Conversation length: {len(test_conversation)} chars")
test_tokenized = prm_tokenizer(test_conversation, return_tensors="pt", padding=False)
test_input_ids = test_tokenized['input_ids'].to(prm_model.device)
test_attention_mask = test_tokenized['attention_mask'].to(prm_model.device)
print(f"  Input IDs shape: {test_input_ids.shape}")

with torch.no_grad():
    test_outputs = prm_model(
        input_ids=test_input_ids,
        attention_mask=test_attention_mask,
        use_cache=False
    )

test_step_sep_id = prm_tokenizer.encode("<extra_0>")[0]
test_token_masks = (test_input_ids == test_step_sep_id)
print(f"  Step separators found: {test_token_masks.sum().item()}")

test_logits = test_outputs.logits if hasattr(test_outputs, 'logits') else test_outputs[0]
print(f"  Logits shape: {test_logits.shape}")
print(f"  Logits contains NaN: {torch.isnan(test_logits).any().item()}")

if not torch.isnan(test_logits).any():
    test_step_rewards = make_step_rewards(test_logits, test_token_masks)
    print(f"✅ Test PRM scores: {test_step_rewards}")
    print(f"   Expected: ~4 scores, each in [0, 1]")
else:
    print(f"❌ Test FAILED: Logits contain NaN")
    print(f"   Logits min/max: {test_logits.min().item():.4f} / {test_logits.max().item():.4f}")



SECTION 4: Implementing PRM scoring functions...

[TEST] Testing PRM scoring with official example format...
  Conversation length: 280 chars
  Input IDs shape: torch.Size([1, 69])


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Step separators found: 4
  Logits shape: torch.Size([1, 69, 2])
  Logits contains NaN: False
✅ Test PRM scores: [[0.515625, 0.53125, 0.5546875, 0.53515625]]
   Expected: ~4 scores, each in [0, 1]


In [50]:
# ============================================================================
# SECTION 5: IMPLEMENT PROF ALGORITHM 1 (FILTERING)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 5: Implementing PROF Algorithm 1...")
print("=" * 60)

def compute_trajectory_prm_score(step_rewards, outcome_reward,
                                lambda_reg=10, H_lambda=30):
    """
    Compute trajectory-level PRM score with step regularization.

    Implements PROF Algorithm 1, Equation 1:
    r_pro_i = [mean(step_rewards) - λ·I(H=1 or H≥H_λ)] · ro,i

    Args:
        step_rewards: List of step-wise PRM scores (each in [0, 1])
        outcome_reward: Outcome reward {-1, 1} (incorrect/correct)
        lambda_reg: Regularization weight λ=10 (Ye et al.)
        H_lambda: Threshold H_λ=30 (Ye et al.)

    Returns:
        trajectory_score: Float representing consistency
    """
    num_steps = len(step_rewards)

    # Trajectory-level score = mean of step-wise rewards
    if num_steps == 0:
        mean_step_reward = 0.0
    else:
        mean_step_reward = sum(step_rewards) / num_steps

    # Step length regularization: penalize trivial (1 step) or very long (>H_λ)
    step_penalty = 0.0
    if num_steps == 1 or num_steps >= H_lambda:
        step_penalty = lambda_reg

    # Final consistency score weighted by outcome reward
    trajectory_score = (mean_step_reward - step_penalty) * outcome_reward

    return trajectory_score


def prof_filter(rollouts, outcome_rewards, prm_model, prm_tokenizer,
                policy_update_size=4, lambda_reg=10, H_lambda=30):
    """
    Implement PROF filtering Algorithm 1 (complete).

    Filters n=8 rollouts down to m=4 based on consistency between
    PRM (process rewards) and ORM (outcome rewards).

    Args:
        rollouts: List of n=8 response texts
        outcome_rewards: List of n=8 outcome rewards {-1, 1}
        prm_model: Loaded PRM for step scoring
        prm_tokenizer: PRM tokenizer
        policy_update_size: Target m=4 (Ye et al.)

    Returns:
        filtered_rollouts: List of m=4 kept responses
        filtered_rewards: List of m=4 corresponding rewards
        filter_stats: Dict with debugging info
    """
    n = len(rollouts)
    n_correct = sum(1 for r in outcome_rewards if r == 1)
    n_incorrect = n - n_correct
    delta = n_correct - n_incorrect

    # Step 1: Compute PRM step scores for each rollout
    all_step_rewards = []
    for response in rollouts:
        step_rewards = compute_step_rewards(response, prm_model, prm_tokenizer)
        all_step_rewards.append(step_rewards)

    # Step 2: Compute trajectory-level consistency scores (Eq. 1)
    trajectory_scores = []
    for step_rewards, outcome in zip(all_step_rewards, outcome_rewards):
        score = compute_trajectory_prm_score(
            step_rewards, outcome, lambda_reg, H_lambda
        )
        trajectory_scores.append(score)

    # Step 3: Separate into correct (G+) and incorrect (G-) groups
    correct_indices = [i for i, r in enumerate(outcome_rewards) if r == 1]
    incorrect_indices = [i for i, r in enumerate(outcome_rewards) if r == -1]

    correct_scores = [(trajectory_scores[i], i) for i in correct_indices]
    incorrect_scores = [(trajectory_scores[i], i) for i in incorrect_indices]

    # Step 4: Calculate removal counts (Eq. 2)
    k_plus = min(n - policy_update_size,
                 int((delta + n - policy_update_size) / 2))
    k_minus = n - policy_update_size - k_plus

    # Step 5: Rank and filter
    # G+ (correct): Keep highest PRM scores (best reasoning among correct)
    # G- (incorrect): Keep lowest PRM scores (most obviously wrong)
    correct_ranked = sorted(correct_scores, key=lambda x: x[0], reverse=True)
    incorrect_ranked = sorted(incorrect_scores, key=lambda x: x[0], reverse=False)

    # Keep top from each group
    kept_correct_indices = [idx for _, idx in correct_ranked[:(n_correct - k_plus)]]
    kept_incorrect_indices = [idx for _, idx in incorrect_ranked[:(n_incorrect - k_minus)]]

    filtered_indices = kept_correct_indices + kept_incorrect_indices

    # Step 6: Build filtered outputs
    filtered_rollouts = [rollouts[i] for i in filtered_indices]
    filtered_rewards = [outcome_rewards[i] for i in filtered_indices]

    filter_stats = {
        'original_count': n,
        'kept_count': len(filtered_indices),
        'correct_removed': k_plus,
        'incorrect_removed': k_minus,
        'correct_kept': len(kept_correct_indices),
        'incorrect_kept': len(kept_incorrect_indices),
        'avg_prm_score_kept': sum(trajectory_scores[i] for i in filtered_indices) / len(filtered_indices) if filtered_indices else 0,
    }

    return filtered_rollouts, filtered_rewards, filter_stats

# Test filtering on dummy data
test_rollouts = ["Response A", "Response B", "Response C", "Response D",
                 "Response E", "Response F", "Response G", "Response H"]
test_outcomes = [1, 1, 1, 1, -1, -1, -1, -1]  # 4 correct, 4 incorrect

filtered, filtered_rewards, stats = prof_filter(
    test_rollouts, test_outcomes, prm_model, prm_tokenizer
)

print(f"\n✅ PROF Filtering Test:")
print(f"   Original: {len(test_rollouts)} rollouts")
print(f"   Filtered: {len(filtered)} rollouts (should be 4)")
print(f"   Stats: {stats}")

assert len(filtered) == 4, "PROF should keep m=4 rollouts!"
print("✅ PROF filtering validation passed!")



SECTION 5: Implementing PROF Algorithm 1...
  [DEBUG] Output type: <class 'transformers.modeling_outputs.TokenClassifierOutput'>
  [DEBUG] Output keys: odict_keys(['logits'])
  [DEBUG] Using outputs.logits
  [DEBUG] Logits shape: torch.Size([1, 29, 2])
  [DEBUG] Logits dtype: torch.bfloat16
  [DEBUG] Logits device: cuda:0
  [DEBUG] Logits contains NaN: False
  [DEBUG] Logits contains Inf: False
  [DEBUG] Logits min/max: -0.9805 / 0.8711
  [DEBUG] Step separator ID: 151651
  [DEBUG] Number of step separators found: 1
  [DEBUG] Output type: <class 'transformers.modeling_outputs.TokenClassifierOutput'>
  [DEBUG] Output keys: odict_keys(['logits'])
  [DEBUG] Using outputs.logits
  [DEBUG] Logits shape: torch.Size([1, 29, 2])
  [DEBUG] Logits dtype: torch.bfloat16
  [DEBUG] Logits device: cuda:0
  [DEBUG] Logits contains NaN: False
  [DEBUG] Logits contains Inf: False
  [DEBUG] Logits min/max: -0.9805 / 0.8711
  [DEBUG] Step separator ID: 151651
  [DEBUG] Number of step separators found: 1

In [51]:
# ============================================================================
# SECTION 6: LOAD TRAINING DATA
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 6: Loading training data...")
print("=" * 60)

# NuminaMath dataset (per Ye et al. paper)
# If not available, use GSM8K as proxy
try:
    train_dataset = load_dataset("AI-MO/NuminaMath-CoT", split="train")
    print("✅ Loaded NuminaMath-CoT dataset")
except:
    print("⚠️  NuminaMath not found, using GSM8K as proxy")
    train_dataset = load_dataset("openai/gsm8k", "main", split="train")

# Format for math reasoning
def format_prompt(example):
    """Convert dataset example to Qwen2.5-Math format"""
    # Format: "Question: {problem}\nAnswer:"
    if "problem" in example:
        problem = example["problem"]
    elif "question" in example:
        problem = example["question"]
    else:
        raise KeyError("No problem/question field in dataset")

    return f"Question: {problem}\nAnswer:"

# Prepare prompts (1024 per Ye et al.)
train_prompts = [format_prompt(ex) for ex in train_dataset.select(range(min(1024, len(train_dataset))))]
print(f"✅ Loaded {len(train_prompts)} training prompts")
print(f"   Example prompt:\n   {train_prompts[0][:150]}...")



SECTION 6: Loading training data...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

...

data/train-00004-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/166k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/859494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Loaded NuminaMath-CoT dataset
✅ Loaded 1024 training prompts
   Example prompt:
   Question: Consider the terms of an arithmetic sequence: $-\frac{1}{3}, y+2, 4y, \ldots$. Solve for $y$.
Answer:...


In [52]:
# ============================================================================
# SECTION 7: IMPLEMENT BINARY OUTCOME REWARD (VERIFIER)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 7: Implementing binary outcome reward...")
print("=" * 60)

def extract_answer(text):
    """Extract final numerical answer from model output"""
    # Look for patterns like "The answer is X" or "####X" (GSM8K format)
    patterns = [
        r'####\s*([0-9,\.]+)',           # GSM8K format
        r'[Tt]he answer is[:\s]*([0-9,\.]+)',
        r'[Aa]nswer[:\s]*([0-9,\.]+)',
        r'\\boxed\{([^}]+)\}',            # LaTeX boxed
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1).replace(',', '').strip()

    # Fallback: last number in text
    numbers = re.findall(r'[0-9,\.]+', text)
    return numbers[-1].replace(',', '').strip() if numbers else None

def binary_outcome_reward(prompt, response, ground_truth):
    """
    Binary outcome reward: +1 if correct, -1 if incorrect
    Per Ye et al.: ro ∈ {-1, +1}
    """
    predicted = extract_answer(response)

    if predicted is None:
        return -1.0  # No answer found

    # Normalize both answers
    try:
        pred_num = float(predicted)
        gt_num = float(ground_truth)
        is_correct = abs(pred_num - gt_num) < 1e-4
    except ValueError:
        # String comparison fallback
        is_correct = predicted.strip().lower() == str(ground_truth).strip().lower()

    return 1.0 if is_correct else -1.0

# Test on example
test_response = "Let me solve this step by step.\n1. First...\n2. Then...\nThe answer is 42."
test_reward = binary_outcome_reward("", test_response, "42")
print(f"✅ Test reward: {test_reward} (should be 1.0)")


SECTION 7: Implementing binary outcome reward...
✅ Test reward: 1.0 (should be 1.0)


In [58]:
# ============================================================================
# SECTION 8: CONFIGURE GRPO TRAINER (BASELINE)
# ============================================================================
print("\n" + "=" * 60)
print("SECTION 8: Configuring GRPO trainer...")
print("=" * 60)
from trl import GRPOConfig, GRPOTrainer

# Hyperparameters from Ye et al. Section 4.1
grpo_config = GRPOConfig(
    output_dir="./baseline_grpo_checkpoints",
    num_train_epochs=1,
    max_steps=1,                          # ← move here for smoke test
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-6,
    optim="adamw_torch",
    warmup_steps=10,
    num_generations=8,
    temperature=1.0,
    beta=0.001,
    epsilon=0.2,
    epsilon_high=0.28,
    max_completion_length=4096,
    max_prompt_length=512,
    logging_steps=10,
    save_steps=200,
    save_total_limit=5,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
)
print("✅ GRPO config loaded:")
print(f"   Effective batch size: {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")
print(f"   Generations per prompt: {grpo_config.num_generations}")
print(f"   Learning rate: {grpo_config.learning_rate}")


SECTION 8: Configuring GRPO trainer...
✅ GRPO config loaded:
   Effective batch size: 16
   Generations per prompt: 8
   Learning rate: 1e-06


In [59]:
# ============================================================================
# SECTION 9: SMOKE TEST - Single Forward + Backward Pass
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 9: RUNNING SMOKE TEST...")
print("=" * 60)
print("⚠️  CRITICAL: This must pass before proceeding to Day 2")
print("=" * 60)

# Create minimal trainer for smoke test
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    processing_class=tokenizer, 
    reward_funcs=lambda prompts, responses: [binary_outcome_reward(p, r, "42") for p, r in zip(prompts, responses)],
    train_dataset=train_prompts[:10],  # Only 10 samples for smoke test
)

# Clear VRAM before test
torch.cuda.empty_cache()
initial_memory = torch.cuda.memory_allocated() / 1e9

# Run 1 training step
print("\n🔥 Starting smoke test (1 training step)...")
print("   This will:")
print("   1. Generate n=8 rollouts per prompt")
print("   2. Compute rewards")
print("   3. Run GRPO loss backward pass")
print("   4. Update weights")
print()

try:
    trainer.train()

    peak_memory = torch.cuda.max_memory_allocated() / 1e9
    print(f"\n{'='*60}")
    print(f"✅ SMOKE TEST PASSED")
    print(f"{'='*60}")
    print(f"   Initial VRAM: {initial_memory:.2f} GB")
    print(f"   Peak VRAM: {peak_memory:.2f} GB")
    print(f"   Available headroom: {16 - peak_memory:.2f} GB")
    print(f"{'='*60}")

    if peak_memory > 15.0:
        print(f"⚠️  WARNING: Very tight memory ({peak_memory:.2f}/16 GB)")
        print(f"   Consider: reduce batch_size or num_generations")
    else:
        print(f"✅ Memory footprint looks good!")

    print(f"\n{'='*60}")
    print(f"🎉 DAY 1 COMPLETE - ALL GATES PASSED")
    print(f"{'='*60}")
    print(f"✅ Policy model loaded (~7GB)")
    print(f"✅ PRM model loaded (~3GB)")
    print(f"✅ PRM scoring works")
    print(f"✅ PROF filtering works (8 → 4)")
    print(f"✅ GRPO training loop works")
    print(f"✅ Peak VRAM: {peak_memory:.2f} GB (safe margin)")
    print(f"\n🚀 READY FOR DAY 2: Launch full training!")
    print(f"{'='*60}")

except RuntimeError as e:
    if "out of memory" in str(e):
        print(f"\n{'='*60}")
        print(f"❌ SMOKE TEST FAILED: OOM")
        print(f"{'='*60}")
        print(f"   Peak VRAM before crash: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
        print(f"\n🔧 Troubleshooting:")
        print(f"   1. Reduce num_generations: 8 → 4")
        print(f"   2. Reduce gradient_accumulation_steps: 16 → 8")
        print(f"   3. Reduce max_new_tokens: 4096 → 2048")
        print(f"   4. Enable load_in_4bit=True (last resort)")
        print(f"{'='*60}")
        raise
    else:
        print(f"\n{'='*60}")
        print(f"❌ SMOKE TEST FAILED: {e}")
        print(f"{'='*60}")
        raise


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



SECTION 9: RUNNING SMOKE TEST...
⚠️  CRITICAL: This must pass before proceeding to Day 2

🔥 Starting smoke test (1 training step)...
   This will:
   1. Generate n=8 rollouts per prompt
   2. Compute rewards
   3. Run GRPO loss backward pass
   4. Update weights



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


TypeError: string indices must be integers, not 'str'